
Travelagent-Lakehouse Kafka Producers
=====================================
Two realistic event producers for Travel Agent business:
1. Flight Status Events (delays, cancellations, diversions)
2. Booking Events (created, confirmed, failed, cancelled)



In [3]:
import time
import json
import random
import uuid
from datetime import datetime, timedelta, timezone
from faker import Faker
from kafka import KafkaProducer
import threading
from queue import Queue

In [4]:
# CONFIGURATION


KAFKA_BOOTSTRAP_SERVERS = ['kafka1:9092', 'kafka2:9092', 'kafka3:9092']
FLIGHT_STATUS_TOPIC = 'flight-status-events'
BOOKING_EVENTS_TOPIC = 'booking-events'


In [5]:
# Realistic airline data
AIRLINES = {
    'AA': {'name': 'American Airlines', 'country': 'USA'},
    'UA': {'name': 'United Airlines', 'country': 'USA'},
    'DL': {'name': 'Delta Air Lines', 'country': 'USA'},
    'BA': {'name': 'British Airways', 'country': 'UK'},
    'AF': {'name': 'Air France', 'country': 'France'},
    'LH': {'name': 'Lufthansa', 'country': 'Germany'},
    'EK': {'name': 'Emirates', 'country': 'UAE'},
    'QR': {'name': 'Qatar Airways', 'country': 'Qatar'},
    'SQ': {'name': 'Singapore Airlines', 'country': 'Singapore'},
    'EY': {'name': 'Etihad Airways', 'country': 'UAE'}
}

# Major airports
AIRPORTS = {
    'JFK': {'name': 'John F Kennedy Intl', 'city': 'New York', 'country': 'USA'},
    'LAX': {'name': 'Los Angeles Intl', 'city': 'Los Angeles', 'country': 'USA'},
    'ORD': {'name': "O'Hare Intl", 'city': 'Chicago', 'country': 'USA'},
    'LHR': {'name': 'Heathrow', 'city': 'London', 'country': 'UK'},
    'CDG': {'name': 'Charles de Gaulle', 'city': 'Paris', 'country': 'France'},
    'FRA': {'name': 'Frankfurt', 'city': 'Frankfurt', 'country': 'Germany'},
    'DXB': {'name': 'Dubai Intl', 'city': 'Dubai', 'country': 'UAE'},
    'SIN': {'name': 'Changi', 'city': 'Singapore', 'country': 'Singapore'},
    'HND': {'name': 'Tokyo Haneda', 'city': 'Tokyo', 'country': 'Japan'},
    'AUH': {'name': 'Abu Dhabi Intl', 'city': 'Abu Dhabi', 'country': 'UAE'}
}

# Aircraft types
AIRCRAFT_TYPES = ['B737', 'B738', 'B77W', 'B787', 'A320', 'A321', 'A330', 'A350']

# Flight status distribution
FLIGHT_STATUS_WEIGHTS = {
    'ON_TIME': 0.70,
    'DELAYED': 0.20,
    'CANCELLED': 0.07,
    'DIVERTED': 0.03
}

# Disruption reasons
DISRUPTION_REASONS = ['WEATHER', 'TECHNICAL', 'CREW', 'AIR_TRAFFIC']

# Booking events
BOOKING_EVENTS = {
    'BOOKING_CREATED': 0.40,
    'BOOKING_CONFIRMED': 0.35,
    'BOOKING_FAILED': 0.15,
    'BOOKING_CANCELLED': 0.10
}

# Failure reasons
FAILURE_REASONS = ['PAYMENT_FAILED', 'SEAT_UNAVAILABLE', 'TIMEOUT', 'INVALID_DATA']

# Booking channels
CHANNELS = ['WEB', 'MOBILE', 'API', 'CALL_CENTER']

In [ ]:
class BaseProducer:
    """
    Base class for Kafka producers.
    Handles initialization, logging, and graceful shutdown
    with thread-safe output using a shared queue.
    """

    def __init__(self, name, kafka_servers, topic, output_queue):
        self.name = name
        self.topic = topic
        self.output_queue = output_queue

        self.fake = Faker()
        self.running = True
        self.producer = None

        self._initialize_producer(kafka_servers)

    def _initialize_producer(self, kafka_servers):
        try:
            self.producer = KafkaProducer(
                bootstrap_servers=kafka_servers,
                value_serializer=lambda value: json.dumps(value).encode("utf-8"),
                max_block_ms=5000
            )
            self.log(f"{self.name} initialized. Topic: {self.topic}")
        except Exception as exc:
            self.log(f"Failed to initialize producer: {exc}")
            raise

    def log(self, message):
        """
        Push log messages to the shared output queue
        to ensure thread-safe logging.
        """
        self.output_queue.put(f"[{self.name}] {message}")

    def stop(self):
        """
        Stop the producer gracefully and release resources.
        """
        self.running = False

        if self.producer is None:
            return

        try:
            self.producer.close()
            self.log("Producer closed successfully")
        except Exception:
            pass


In [ ]:
class FlightStatusProducer(BaseProducer):
    """
    Produces flight status events to Kafka.
    Each event represents the lifecycle and current status of a flight.
    """

    def __init__(self, kafka_servers, topic, output_queue):
        super().__init__("FlightStatus", kafka_servers, topic, output_queue)

    def generate_event(self):
        """
        Generate a single flight status event with realistic timing,
        status distribution, and operational metadata.
        """
        event_time = datetime.now(timezone.utc)

        airline_iata = random.choice(list(AIRLINES.keys()))
        source_airport = random.choice(list(AIRPORTS.keys()))
        destination_airport = random.choice(
            [code for code in AIRPORTS.keys() if code != source_airport]
        )

        flight_number = f"{airline_iata}{random.randint(100, 9999)}"

        scheduled_departure = event_time + timedelta(hours=random.uniform(0, 12))
        flight_duration = timedelta(hours=random.uniform(1, 15))
        scheduled_arrival = scheduled_departure + flight_duration

        status = random.choices(
            list(FLIGHT_STATUS_WEIGHTS.keys()),
            weights=list(FLIGHT_STATUS_WEIGHTS.values())
        )[0]

        actual_departure = None
        actual_arrival = None
        delay_minutes = None
        disruption_reason = None
        gate_change = False

        if status == "ON_TIME":
            actual_departure = scheduled_departure + timedelta(minutes=random.randint(-5, 5))
            actual_arrival = scheduled_arrival + timedelta(minutes=random.randint(-10, 10))
            delay_minutes = 0
            gate_change = random.random() < 0.1

        elif status == "DELAYED":
            delay_minutes = random.choices(
                [15, 30, 60, 120, 180],
                weights=[0.3, 0.3, 0.2, 0.15, 0.05]
            )[0]
            actual_departure = scheduled_departure + timedelta(minutes=delay_minutes)
            actual_arrival = scheduled_arrival + timedelta(minutes=delay_minutes)
            disruption_reason = random.choice(DISRUPTION_REASONS)
            gate_change = random.random() < 0.3

        elif status == "CANCELLED":
            disruption_reason = random.choice(DISRUPTION_REASONS)

        else:
            actual_departure = scheduled_departure + timedelta(minutes=random.randint(-10, 30))
            delay_minutes = random.randint(60, 180)
            actual_arrival = scheduled_arrival + timedelta(minutes=delay_minutes)
            disruption_reason = random.choice(
                ["WEATHER", "TECHNICAL", "MEDICAL_EMERGENCY"]
            )
            gate_change = True

        return {
            "event_id": str(uuid.uuid4()),
            "event_time": event_time.isoformat(),
            "flight_number": flight_number,
            "trip_id": f"TRIP-{uuid.uuid4().hex[:12].upper()}",
            "airline_id": airline_iata,
            "aircraft_id": f"{random.choice(AIRCRAFT_TYPES)}-{random.randint(1000, 9999)}",
            "airport_src_id": source_airport,
            "airport_dst_id": destination_airport,
            "scheduled_departure": scheduled_departure.isoformat(),
            "actual_departure": actual_departure.isoformat() if actual_departure else None,
            "scheduled_arrival": scheduled_arrival.isoformat(),
            "actual_arrival": actual_arrival.isoformat() if actual_arrival else None,
            "flight_status": status,
            "delay_minutes": delay_minutes,
            "gate_change": gate_change,
            "disruption_reason": disruption_reason,
            "source_system": "FLIGHT_OPS",
            "ingestion_time": datetime.now(timezone.utc).isoformat()
        }

    def run(self, interval=15, batch_size=50):
        """
        Continuously produce batches of flight status events
        at a fixed interval.
        """
        self.log(f"Starting producer (interval={interval}s, batch_size={batch_size})")
        iteration = 0

        try:
            while self.running:
                iteration += 1
                start_time = time.time()

                self.log(f"Iteration {iteration} started at {datetime.now().strftime('%H:%M:%S')}")

                events = [self.generate_event() for _ in range(batch_size)]

                sent_count = 0
                status_summary = {
                    "ON_TIME": 0,
                    "DELAYED": 0,
                    "CANCELLED": 0,
                    "DIVERTED": 0
                }

                for event in events:
                    try:
                        self.producer.send(self.topic, value=event)
                        sent_count += 1
                        status_summary[event["flight_status"]] += 1
                    except Exception as exc:
                        self.log(f"Failed to send event: {exc}")

                self.producer.flush()
                elapsed_time = time.time() - start_time

                self.log(f"Sent {sent_count} events in {elapsed_time:.2f} seconds")
                self.log(
                    f"Status summary - "
                    f"On-time: {status_summary['ON_TIME']}, "
                    f"Delayed: {status_summary['DELAYED']}, "
                    f"Cancelled: {status_summary['CANCELLED']}, "
                    f"Diverted: {status_summary['DIVERTED']}"
                )

                sleep_time = max(0, interval - elapsed_time)
                if sleep_time > 0:
                    time.sleep(sleep_time)

        except Exception as exc:
            self.log(f"Producer stopped due to error: {exc}")
        finally:
            self.stop()


In [ ]:
class BookingEventsProducer(BaseProducer):
    """
    Produces booking-related events such as creation, confirmation,
    failure, and cancellation.
    """

    def __init__(self, kafka_servers, topic, output_queue):
        super().__init__("BookingEvents", kafka_servers, topic, output_queue)

    def generate_event(self):
        """
        Generate a single booking event with realistic values
        and weighted event distribution.
        """
        event_time = datetime.now(timezone.utc)

        booking_event = random.choices(
            list(BOOKING_EVENTS.keys()),
            weights=list(BOOKING_EVENTS.values())
        )[0]

        booking_id = f"BKG-{uuid.uuid4().hex[:10].upper()}"
        trip_id = f"TRIP-{uuid.uuid4().hex[:12].upper()}"
        customer_id = f"CUST-{random.randint(10000, 99999)}"

        airline_iata = random.choice(list(AIRLINES.keys()))
        flight_number = f"{airline_iata}{random.randint(100, 9999)}"

        hotel_id = (
            f"HTL-{random.randint(1000, 9999)}"
            if random.random() < 0.3
            else None
        )

        base_price = random.uniform(200, 3000)
        if hotel_id:
            base_price += random.uniform(100, 800)

        failure_reason = None
        booking_value = 0.0

        if booking_event == "BOOKING_FAILED":
            failure_reason = random.choice(FAILURE_REASONS)

        elif booking_event == "BOOKING_CANCELLED":
            failure_reason = "CUSTOMER_INITIATED"

        else:
            booking_value = round(base_price, 2)

        return {
            "event_id": str(uuid.uuid4()),
            "event_time": event_time.isoformat(),
            "booking_id": booking_id,
            "trip_id": trip_id,
            "customer_id": customer_id,
            "flight_number": flight_number,
            "hotel_id": hotel_id,
            "booking_event": booking_event,
            "failure_reason": failure_reason,
            "booking_channel": random.choice(CHANNELS),
            "booking_value": booking_value,
            "source_system": "BOOKING_ENGINE",
            "ingestion_time": datetime.now(timezone.utc).isoformat()
        }

    def run(self, interval=15, batch_size=100):
        """
        Continuously produce booking events in batches
        at a fixed interval.
        """
        self.log(f"Starting producer (interval={interval}s, batch_size={batch_size})")
        iteration = 0

        try:
            while self.running:
                iteration += 1
                start_time = time.time()

                self.log(
                    f"Iteration {iteration} started at "
                    f"{datetime.now().strftime('%H:%M:%S')}"
                )

                events = [self.generate_event() for _ in range(batch_size)]

                sent_count = 0
                event_summary = {event: 0 for event in BOOKING_EVENTS.keys()}
                total_revenue = 0.0

                for event in events:
                    try:
                        self.producer.send(self.topic, value=event)
                        sent_count += 1
                        event_summary[event["booking_event"]] += 1
                        total_revenue += event["booking_value"]
                    except Exception as exc:
                        self.log(f"Failed to send event: {exc}")

                self.producer.flush()
                elapsed_time = time.time() - start_time

                self.log(f"Sent {sent_count} events in {elapsed_time:.2f} seconds")
                self.log(
                    f"Summary - "
                    f"Created: {event_summary['BOOKING_CREATED']}, "
                    f"Confirmed: {event_summary['BOOKING_CONFIRMED']}, "
                    f"Failed: {event_summary['BOOKING_FAILED']}, "
                    f"Cancelled: {event_summary['BOOKING_CANCELLED']}"
                )
                self.log(f"Total revenue: {total_revenue:,.2f}")

                sleep_time = max(0, interval - elapsed_time)
                if sleep_time > 0:
                    time.sleep(sleep_time)

        except Exception as exc:
            self.log(f"Producer stopped due to error: {exc}")
        finally:
            self.stop()


In [ ]:
class ParallelProducerManager:
    """
    Manages multiple Kafka producers running in parallel.
    Handles thread lifecycle and provides thread-safe output
    using a shared queue.
    """

    def __init__(self):
        self.output_queue = Queue()
        self.producers = []
        self.threads = []
        self.running = True

    def start(self):
        """
        Start all producers in parallel and keep the main thread alive
        until interrupted or all producers stop.
        """
        print("\nTravelAgent - Parallel Kafka Producers")
        print("Starting producers...")
        print("To stop execution, interrupt the kernel or press Ctrl+C\n")

        flight_producer = FlightStatusProducer(
            KAFKA_BOOTSTRAP_SERVERS,
            FLIGHT_STATUS_TOPIC,
            self.output_queue
        )

        booking_producer = BookingEventsProducer(
            KAFKA_BOOTSTRAP_SERVERS,
            BOOKING_EVENTS_TOPIC,
            self.output_queue
        )

        self.producers = [flight_producer, booking_producer]

        flight_thread = threading.Thread(
            target=flight_producer.run,
            args=(15, 50),
            daemon=True
        )

        booking_thread = threading.Thread(
            target=booking_producer.run,
            args=(15, 100),
            daemon=True
        )

        self.threads = [flight_thread, booking_thread]

        flight_thread.start()
        time.sleep(1)
        booking_thread.start()

        output_thread = threading.Thread(
            target=self._print_output,
            daemon=True
        )
        output_thread.start()

        print("All producers started successfully\n")

        try:
            while self.running:
                time.sleep(0.5)

                if not any(thread.is_alive() for thread in self.threads):
                    print("\nAll producer threads have stopped")
                    break

        except KeyboardInterrupt:
            print("\nExecution interrupted by user")
            self.stop()

        except Exception as exc:
            print(f"\nUnexpected error occurred: {exc}")
            self.stop()

    def _print_output(self):
        """
        Continuously read log messages from the shared queue
        and print them in a thread-safe manner.
        """
        while self.running:
            try:
                message = self.output_queue.get(timeout=1)
                print(message)
            except Exception:
                continue

    def stop(self):
        """
        Stop all running producers gracefully.
        """
        self.running = False

        for producer in self.producers:
            producer.stop()

        print("All producers stopped\n")


In [ ]:
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    manager = ParallelProducerManager()
    manager.start()


🏨 TRAVELAGENT - DUAL KAFKA PRODUCERS
🚀 Starting both producers automatically...
⚠️  To stop: Click 'Interrupt Kernel' button or press 'I' twice

[FlightStatus] ✅ FlightStatus initialized → Topic: flight-status-events✅ Both producers started!


[BookingEvents] ✅ BookingEvents initialized → Topic: booking-events
[FlightStatus] 🚀 Starting stream (interval=15s, batch=50)
[FlightStatus] ==================================================
[FlightStatus] 📡 Iteration #1 - 16:03:20
[FlightStatus] ✅ Sent 50 events in 0.94s
[FlightStatus] 📊 On-time:39 Delayed:6 Cancelled:3 Diverted:2
[BookingEvents] 🚀 Starting stream (interval=15s, batch=100)
[BookingEvents] ==================================================
[BookingEvents] 🎫 Iteration #1 - 16:03:21
[BookingEvents] ✅ Sent 100 events in 0.29s
[BookingEvents] 📊 Created:46 Confirmed:22 Failed:18 Cancelled:14
[BookingEvents] 💰 Revenue: $118,531.32
[FlightStatus] ==================================================
[FlightStatus] 📡 Iteration #2 - 16:03:

  Using cached faker-38.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached kafka_python-2.3.0-py2.py3-none-any.whl.metadata (10.0 kB)
Using cached faker-38.2.0-py3-none-any.whl (2.0 MB)
Using cached kafka_python-2.3.0-py2.py3-none-any.whl (326 kB)
